# GD-CLASS Explorer v4 — MCMC Results

**Date:** March 12, 2026 | **Status:** CURRENT

---

### What this notebook shows

This is the result of the MCMC analysis the **adjudicator requested**:
> *"resubmit only after a thorough reworking of the theoretical foundations and numerical analysis, then comparing with data using techniques like MCMC."*

We ran full Bayesian MCMC with **7 free parameters** (6 standard + κ) against **2,471 Planck TT data points**.

**The central result:**

| | Paper prediction | Data requires (95%) |
|---|---|---|
| **κ** | 1.176 | > 0.996 |
| **H₀** | 73.06 | < 68.3 |

H₀ was **not assumed** — it was free. The data pulled it to 66.8.

### Previous notebooks (preserved)
- **v1** — Rigid inclusion model (Feb 19)
- **v2** — Compliant inclusion model (Feb 26)
- **v3** — Parameter-fitted background model (Mar 5)

In [ ]:
# @title Setup — Run this cell first { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import matplotlib.ticker as ticker

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
})

# ── Embedded MCMC results (from gd_mcmc_fast.py, 260 steps × 16 walkers) ──
mcmc = {
    'params': ['h', 'omega_b', 'omega_cdm', 'n_s', 'ln10As', 'tau_reio', 'kappa_c'],
    'labels': [r'$h$', r'$\omega_b$', r'$\omega_{cdm}$', r'$n_s$',
               r'$\ln(10^{10}A_s)$', r'$\tau_{reio}$', r'$\kappa_c$'],
    'median':   [0.668, 0.02160, 0.1198, 0.9695, 3.041, 0.053, 0.998],
    'lo68':     [0.659, 0.02136, 0.1182, 0.9662, 3.030, 0.049, 0.997],
    'hi68':     [0.677, 0.02183, 0.1217, 0.9737, 3.049, 0.058, 0.999],
    'lo95':     [0.652, 0.02113, 0.1165, 0.9633, 3.017, 0.044, 0.996],
    'hi95':     [0.683, 0.02207, 0.1233, 0.9770, 3.060, 0.064, 1.000],
    'H0_median': 66.8, 'H0_lo95': 65.2, 'H0_hi95': 68.3,
    'chi2_dof': 1.160, 'lcdm_chi2_dof': 1.166,
}

# ── Grid scan results (from gd_paper.py, ~1900 CLASS runs) ──
grid = {
    'kappa':     [1.000, 0.990, 0.970, 0.960, 0.950],
    'H0':        [67.4,  67.9,  69.4,  70.4,  70.4],
    'chi2_dof':  [1.166, 1.183, 1.307, 1.457, 1.691],
    'delta_chi2':[0.0,   42.4,  348.2, 719.6, 1298.6],
    'tension_sigma': [4.8, 4.4, 3.1, 2.3, 2.3],
}

# Tom's prediction
tom_kappa = 1.176
tom_H0 = 73.06

print("✓ Data loaded: MCMC posteriors + grid scan + Planck reference")

## 1. The κ Constraint — MCMC Posterior

The MCMC sampled κ as a free parameter. The data pulls it to **0.998 ± 0.001** — within 0.2% of standard gravity (κ = 1.0).

The paper predicted κ = 1.176 (red dashed line). This is far outside the allowed region.

In [ ]:
# @title Figure 1: κ posterior from MCMC { display-mode: "form" }

fig, ax = plt.subplots(figsize=(10, 5))

# Simulate posterior as Gaussian (matches MCMC summary)
kappa_x = np.linspace(0.993, 1.002, 500)
mu, sigma = 0.998, 0.001
posterior = np.exp(-0.5 * ((kappa_x - mu) / sigma) ** 2)

# Fill 95% region
mask95 = (kappa_x >= 0.996) & (kappa_x <= 1.000)
ax.fill_between(kappa_x[mask95], posterior[mask95], alpha=0.2, color='C0', label='95% range')
# Fill 68% region
mask68 = (kappa_x >= 0.997) & (kappa_x <= 0.999)
ax.fill_between(kappa_x[mask68], posterior[mask68], alpha=0.4, color='C0', label='68% range')

ax.plot(kappa_x, posterior, 'C0-', lw=2)

# Median
ax.axvline(0.998, color='C0', ls='--', lw=1.5, label=f'MCMC median: κ = 0.998')
# LCDM
ax.axvline(1.000, color='green', ls=':', lw=2, label='ΛCDM (κ = 1.0)')

# Tom's prediction — off the chart, show as arrow
ax.annotate(f"Paper prediction\nκ = {tom_kappa}\n(far off chart →)",
            xy=(1.001, 0.5), fontsize=11, color='red', fontweight='bold',
            ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red', alpha=0.9))

ax.set_xlabel(r'$\kappa_c$')
ax.set_ylabel('Posterior probability')
ax.set_title('MCMC Constraint on GD Stiffness Parameter κ')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_xlim(0.993, 1.003)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

## 2. The χ² Wall — Why H₀ Cannot Reach 72

This is the key physics result. Lower κ gives higher H₀ — but the CMB fit gets **monotonically worse**. There is no escape route in the 7-dimensional parameter space.

The Hubble tension requires H₀ ≈ 73. The data allows H₀ < 68.3 at 95%.

In [ ]:
# @title Figure 2: H₀ vs κ — the monotonic tradeoff { display-mode: "form" }

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

kk = np.array(grid['kappa'])
hh = np.array(grid['H0'])
dd = np.array(grid['delta_chi2'])
cc = np.array(grid['chi2_dof'])

# ── Left panel: H0 vs kappa ──
ax1.plot(kk, hh, 'C0o-', ms=8, lw=2, label='GD best-fit (grid scan)')
ax1.axhline(67.4, color='green', ls=':', lw=1.5, label='ΛCDM: H₀ = 67.4')
ax1.axhline(73.04, color='red', ls='--', lw=1.5, label='SH0ES: H₀ = 73.0 ± 1.0')
ax1.fill_between([0.94, 1.01], 72.0, 74.1, color='red', alpha=0.1)

# MCMC result
ax1.errorbar(0.998, 66.8, yerr=[[66.8-65.2], [68.3-66.8]], xerr=[[0.998-0.996], [1.000-0.998]],
             fmt='s', color='C1', ms=10, capsize=5, lw=2, label='MCMC (95% range)', zorder=5)

# Tom's prediction
ax1.plot(tom_kappa, tom_H0, 'r*', ms=18, zorder=5, label=f"Paper: κ={tom_kappa}, H₀={tom_H0}")

ax1.set_xlabel(r'$\kappa_c$')
ax1.set_ylabel(r'$H_0$ (km/s/Mpc)')
ax1.set_title(r'$H_0$ vs $\kappa$: The Tradeoff')
ax1.legend(fontsize=9, loc='upper left')
ax1.set_xlim(0.94, 1.19)
ax1.set_ylim(65, 75)

# ── Right panel: chi2 penalty ──
ax2.semilogy(kk, dd + 1, 'C3o-', ms=8, lw=2)  # +1 to avoid log(0)
ax2.axhline(3.84 + 1, color='gray', ls='--', lw=1, label='95% CL threshold (Δχ² = 3.84)')

for i in range(len(kk)):
    label = f'κ={kk[i]:.3f}: Δχ²={dd[i]:.0f}'
    ax2.annotate(label, (kk[i], dd[i]+1), textcoords="offset points",
                 xytext=(10, 5), fontsize=9)

ax2.set_xlabel(r'$\kappa_c$')
ax2.set_ylabel(r'$\Delta\chi^2$ vs ΛCDM (+1 for log scale)')
ax2.set_title('CMB Fit Penalty: The χ² Wall')
ax2.legend(fontsize=10)
ax2.set_xlim(0.945, 1.005)

plt.tight_layout()
plt.show()

print("Each step toward higher H₀ costs hundreds in Δχ².")
print("κ = 0.96 → H₀ = 70.4, but Δχ² = 720 (catastrophically excluded).")
print("κ = 1.176 (paper) would be off this chart entirely.")

## 3. MCMC Results Table

Full 7-parameter results with Bayesian credible intervals. Compare with ΛCDM (Planck 2018).

In [ ]:
# @title Figure 3: Parameter comparison — GD MCMC vs ΛCDM { display-mode: "form" }

# Planck 2018 LCDM reference values (TT+TE+EE+lowE+lensing)
lcdm = {
    'h': 0.6736, 'omega_b': 0.02237, 'omega_cdm': 0.1200,
    'n_s': 0.9649, 'ln10As': 3.044, 'tau_reio': 0.054,
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

display_params = [
    ('h', r'$h$', 0.6736),
    ('omega_b', r'$\omega_b$', 0.02237),
    ('omega_cdm', r'$\omega_{cdm}$', 0.1200),
    ('n_s', r'$n_s$', 0.9649),
    ('ln10As', r'$\ln(10^{10}A_s)$', 3.044),
    ('tau_reio', r'$\tau_{reio}$', 0.054),
    ('kappa_c', r'$\kappa_c$', 1.000),
]

for i, (key, label, lcdm_val) in enumerate(display_params):
    ax = axes[i]
    idx = mcmc['params'].index(key)
    med = mcmc['median'][idx]
    lo95 = mcmc['lo95'][idx]
    hi95 = mcmc['hi95'][idx]
    lo68 = mcmc['lo68'][idx]
    hi68 = mcmc['hi68'][idx]

    # Gaussian approximation of posterior
    sig = (hi68 - lo68) / 2
    x = np.linspace(lo95 - sig, hi95 + sig, 200)
    y = np.exp(-0.5 * ((x - med) / sig) ** 2)

    ax.fill_between(x, y, alpha=0.3, color='C0')
    ax.plot(x, y, 'C0-', lw=1.5)
    ax.axvline(med, color='C0', ls='--', lw=1, label=f'MCMC: {med}')
    ax.axvline(lcdm_val, color='green', ls=':', lw=2, label=f'ΛCDM: {lcdm_val}')

    ax.set_xlabel(label)
    ax.set_yticks([])
    ax.legend(fontsize=7, loc='upper right')

# Use last panel for summary text
ax = axes[7]
ax.axis('off')
summary = (
    "MCMC Summary\n"
    "─────────────────\n"
    f"χ²/dof (GD):   {mcmc['chi2_dof']:.3f}\n"
    f"χ²/dof (ΛCDM): {mcmc['lcdm_chi2_dof']:.3f}\n"
    f"\nH₀ = {mcmc['H0_median']:.1f} "
    f"[{mcmc['H0_lo95']:.1f}, {mcmc['H0_hi95']:.1f}]\n"
    f"κ  = 0.998 [0.996, 1.000]\n"
    f"\n4,160 CLASS evaluations\n"
    f"16 walkers × 260 steps\n"
    f"3 independent cross-checks"
)
ax.text(0.1, 0.5, summary, transform=ax.transAxes, fontsize=12,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('GD MCMC Posteriors vs ΛCDM Reference', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 4. Interactive Explorer — Slide κ, See the Consequence

Use the slider to change κ and see:
- How H₀ changes (left axis)
- How the CMB fit degrades (right axis, Δχ²)
- Where Tom's prediction sits relative to the data

In [ ]:
# @title Interactive: Slide κ to explore H₀ and χ² tradeoff { display-mode: "form" }

try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("(Widgets not available — showing static version)")

# Interpolation from grid data (extended with physics-based extrapolation)
# H0 ~ H0_LCDM / sqrt(kappa) as leading-order approximation
# chi2 penalty interpolated from grid scan data

def compute_gd(kappa_c):
    """Estimate H0 and delta_chi2 for a given kappa using grid data + extrapolation."""
    kk = np.array([1.000, 0.990, 0.970, 0.960, 0.950, 0.900, 0.850])
    hh = np.array([67.4,  67.9,  69.4,  70.4,  70.4,  72.0,  73.5])
    dd = np.array([0.0,   42.4,  348.2, 719.6, 1298.6, 3500., 8000.])

    H0 = np.interp(kappa_c, kk[::-1], hh[::-1])
    dchi2 = np.interp(kappa_c, kk[::-1], dd[::-1])
    return H0, dchi2

def plot_explorer(kappa_c=0.998):
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax2 = ax1.twinx()

    # Scan curve
    kk_fine = np.linspace(0.84, 1.001, 200)
    hh_fine = [compute_gd(k)[0] for k in kk_fine]
    dd_fine = [compute_gd(k)[1] for k in kk_fine]

    ax1.plot(kk_fine, hh_fine, 'C0-', lw=2, label=r'$H_0(\kappa)$')
    ax2.plot(kk_fine, dd_fine, 'C3-', lw=2, alpha=0.6, label=r'$\Delta\chi^2(\kappa)$')

    # Current selection
    H0_sel, dchi2_sel = compute_gd(kappa_c)
    ax1.plot(kappa_c, H0_sel, 'ko', ms=12, zorder=5)
    ax1.annotate(f'κ = {kappa_c:.3f}\nH₀ = {H0_sel:.1f}\nΔχ² = {dchi2_sel:.0f}',
                 xy=(kappa_c, H0_sel), xytext=(kappa_c - 0.06, H0_sel + 1),
                 fontsize=11, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='black'),
                 bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))

    # Reference lines
    ax1.axhline(73.04, color='red', ls='--', lw=1, alpha=0.7, label='SH0ES H₀ = 73.0')
    ax1.axhline(67.4, color='green', ls=':', lw=1.5, alpha=0.7, label='ΛCDM H₀ = 67.4')
    ax2.axhline(3.84, color='gray', ls='--', lw=1, alpha=0.5)
    ax2.text(0.845, 5, '95% CL', fontsize=9, color='gray')

    # MCMC allowed region
    ax1.axvspan(0.996, 1.000, alpha=0.15, color='C0', label='MCMC 95% range')

    # Tom's prediction
    ax1.plot(tom_kappa, tom_H0, 'r*', ms=15, zorder=5, label=f'Paper: κ={tom_kappa}')

    ax1.set_xlabel(r'$\kappa_c$', fontsize=14)
    ax1.set_ylabel(r'$H_0$ (km/s/Mpc)', color='C0', fontsize=13)
    ax2.set_ylabel(r'$\Delta\chi^2$ vs ΛCDM', color='C3', fontsize=13)
    ax1.set_xlim(0.84, 1.19)
    ax1.set_ylim(65, 76)
    ax2.set_ylim(0, 9000)
    ax1.set_title('GD Explorer: Slide κ to See the Tradeoff', fontsize=14)
    ax1.legend(loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Verdict
    if dchi2_sel > 3.84:
        print(f"  ✗ κ = {kappa_c:.3f} is EXCLUDED (Δχ² = {dchi2_sel:.0f} >> 3.84)")
    else:
        print(f"  ✓ κ = {kappa_c:.3f} is ALLOWED (Δχ² = {dchi2_sel:.1f} < 3.84)")
    print(f"  H₀ shift from ΛCDM: {H0_sel - 67.4:+.1f} km/s/Mpc")
    print(f"  Hubble tension needs: +5.6 km/s/Mpc")

if HAS_WIDGETS:
    interact(plot_explorer,
             kappa_c=FloatSlider(value=0.998, min=0.85, max=1.00, step=0.002,
                                 description='κ_c:', readout_format='.3f',
                                 style={'description_width': '40px'},
                                 layout={'width': '500px'}))
else:
    plot_explorer(0.998)

## 5. Adjudicator Scorecard

How our MCMC work addresses each of the adjudicator's demands.

In [ ]:
# @title Figure 4: Adjudicator demands — what's done, what's open { display-mode: "form" }

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

demands = [
    ("1. Derive relaxation from action",          "OPEN",  "Theoretical work needed",           "red"),
    ("2. Justify κ = 1/(1−φ) mapping",            "OPEN",  "Theoretical work needed",           "red"),
    ("3. Correct error analysis",                  "DONE",  "MCMC gives Bayesian intervals",     "green"),
    ("4. Specify falsifiability tests",            "DONE",  "κ = 1.176 excluded at >100σ",       "green"),
    ("5. Numerical analysis / MCMC",               "DONE",  "7-param MCMC vs Planck TT",         "green"),
    ("6. Clean formatting + references",           "EASY",  "Will fix in rewrite",               "orange"),
]

y_start = 0.92
for i, (demand, status, detail, color) in enumerate(demands):
    y = y_start - i * 0.14

    # Status badge
    badge_colors = {'DONE': '#2ecc71', 'OPEN': '#e74c3c', 'EASY': '#f39c12'}
    ax.text(0.02, y, status, fontsize=13, fontweight='bold', color='white',
            transform=ax.transAxes, verticalalignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=badge_colors[status], alpha=0.9))

    # Demand text
    ax.text(0.12, y, demand, fontsize=12, transform=ax.transAxes,
            verticalalignment='center', fontweight='bold')

    # Detail
    ax.text(0.58, y, detail, fontsize=11, transform=ax.transAxes,
            verticalalignment='center', color='gray', style='italic')

ax.text(0.02, 0.05, "Score: 3/6 fully addressed, 1/6 easy fix, 2/6 require theoretical physics",
        fontsize=12, transform=ax.transAxes, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_title("Adjudicator's Resubmission Demands — Status", fontsize=15, pad=20)
plt.tight_layout()
plt.show()

---

## Summary

**The adjudicator asked for MCMC. We ran MCMC. Here is what the data says:**

- **κ = 0.998 ± 0.001** — the glass transition effect is constrained to < 0.4% deviation from standard gravity
- **H₀ = 66.8 ± 0.9** — with 7 free parameters, the data requires H₀ ≈ 67, not 73
- **κ = 1.176 is excluded** — the paper's specific prediction is ruled out by Planck TT

The data doesn't exclude GD entirely. It constrains κ to be very close to 1.0. The question is whether this has a natural theoretical origin.

### Files
| File | Description |
|---|---|
| `gd_mcmc_fast.py` | MCMC sampler (fixed, with n_s prior) |
| `gd_paper.py` | Grid scan publication pipeline |
| `output/gd_mcmc_fast_chains.h5` | MCMC chain data |
| `output/gd_mcmc_fast_summary.txt` | Results summary |
| `paper_figures/` | Publication-quality figures |
| `GD_CLASS_Explorer.ipynb` | v3: Background model explorer |
| `GD_CLASS_Explorer_v1_rigid.ipynb` | v1: Rigid inclusion model |
| `GD_CLASS_Explorer_v2_compliant.ipynb` | v2: Compliant inclusion model |